# 01 ? Preprocesamiento de Eráviri

Primera etapa del experimento. Ejecutar todas las celdas en orden. Solo se requiere la biblioteca estándar de Python. No se entrena todav?a ning?n modelo ni se generan traducciones.

Conservamos tildes, negación y sufijos: no stopwords, stemming ni lematizaci?n. Se mantienen IDs, orden y texto original; las particiones nunca se mezclan. Los signos se separan para obtener tokens superficiales. NFC evita diferencias entre representaciones Unicode de una misma tilde.


In [1]:
from pathlib import Path
import sys

# Funciona desde la raíz del proyecto o cualquier subcarpeta del repositorio.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents)
     if (p / 'ENUNCIADO.md').is_file() and (p / 'datos').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Abre el notebook dentro de la carpeta GestionConocimiento.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Raíz del proyecto:', PROJECT_ROOT)


Raíz del proyecto: C:\Users\aml\Desktop\GestionConocimiento


In [2]:
from lexicon.preprocessing import prepare, export_preprocessed
processed, audit = prepare(PROJECT_ROOT)
for split, info in audit['splits'].items():
    print(split, info)


train {'rows': 420, 'unique_sources_normalized': 420, 'vocabulary_size': 59, 'oov_against_train': [], 'lengths': {3: 95, 4: 269, 5: 56}, 'token_frequencies': {'nu': 85, 'kuna': 58, 'pavo': 55, 'yami': 50, 'rilo': 49, 'raku': 48, 'hala': 45, 'miri': 45, 'vonu': 44, 'sova': 44, 'telu': 44, 'lina': 43, 'dali': 42, 'sari': 42, 'moru': 41, 'tora': 41, 'penu': 40, 'beko': 40, 'naku': 38, 'mika': 37, 'kava': 36, 'suni': 35, 'kemi': 34, 'kori': 34, 'tavi': 34, 'nelo': 33, 'pira': 32, 'faro': 32, 'dema': 30, 'zura': 30, 'rima': 29, 'feni': 27, 'lomu': 26, 'sema': 26, 'taku': 26, 'yelo': 25, 'paki': 24, 'vasi': 19, 'nemi': 18, 'vasita': 15, 'koriri': 13, 'nemiri': 12, 'yelori': 11, 'farori': 10, 'lomuta': 9, 'semata': 9, 'takuri': 9, 'yelota': 8, 'takuta': 8, 'vasiri': 8, 'pakiri': 8, 'lomuri': 6, 'pakita': 6, 'demata': 6, 'demari': 5, 'semari': 5, 'farota': 4, 'nemita': 4, 'korita': 4}}
validation {'rows': 120, 'unique_sources_normalized': 120, 'vocabulary_size': 58, 'oov_against_train': [], 'l

## Inspección de original, normalización y tokens
No segmentamos -ta ni -ri automáticamente en esta fase. Su interpretación pertenece al análisis lingüístico posterior; los tokens superficiales se conservan para el baseline y el futuro modelo de n-gramas.


In [3]:
for split, rows in processed.items():
    print(split, rows[0])


train {'id': 'NX-0001', 'naviri': 'vonu pira hala yelo.', 'espanol': 'el viajero elige el agua en la casa.', 'naviri_normalizado': 'vonu pira hala yelo.', 'naviri_tokens': ['vonu', 'pira', 'hala', 'yelo'], 'espanol_normalizado': 'el viajero elige el agua en la casa.', 'espanol_tokens': ['el', 'viajero', 'elige', 'el', 'agua', 'en', 'la', 'casa']}
validation {'id': 'NX-0001', 'naviri': 'telu tavi nemi.', 'espanol': 'el maestro recuerda el alimento.', 'naviri_normalizado': 'telu tavi nemi.', 'naviri_tokens': ['telu', 'tavi', 'nemi'], 'espanol_normalizado': 'el maestro recuerda el alimento.', 'espanol_tokens': ['el', 'maestro', 'recuerda', 'el', 'alimento']}
monolingual {'id': 'NX-0001', 'naviri': 'penu beko hala sema.', 'naviri_normalizado': 'penu beko hala sema.', 'naviri_tokens': ['penu', 'beko', 'hala', 'sema']}
public {'id': 'NX-0001', 'naviri': 'telu suni kuna semari.', 'naviri_normalizado': 'telu suni kuna semari.', 'naviri_tokens': ['telu', 'suni', 'kuna', 'semari']}


## Separación experimental
El vocabulario de referencia se ajusta solo con entrenamiento. El monolingüe podrá alimentar un modelo de lenguaje separado; no se usan traducciones de validaci?n para aprender correspondencias. Los cruces se auditan, sin borrar ni redistribuir ejemplos. La exposición a entradas públicas sin etiquetas debe declararse.


In [4]:
print('Cruces normalizados:', audit['source_overlaps'])
print('Vocabulario ajustado en:', audit['vocabulary_fit_split'])


Cruces normalizados: {'train__validation': 0, 'train__monolingual': 0, 'train__public': 0, 'validation__monolingual': 0, 'validation__public': 0, 'monolingual__public': 28}
Vocabulario ajustado en: train


In [5]:
report = export_preprocessed(PROJECT_ROOT)
print('Exportado a:', PROJECT_ROOT / 'resultados' / 'preprocesamiento')


Exportado a: C:\Users\aml\Desktop\GestionConocimiento\resultados\preprocesamiento


## Próxima etapa
Construir un modelo probabilístico de bigramas/trigramas en Eráviri con el monolingüe, definiendo fronteras de oraci?n y suavizado dentro del modelo. Este cuaderno no incorpora todav?a ese entrenamiento. El módulo traductor se aprenderá después con los pares bilingües.
